# 练习 5：生成器表达式与惰性求值

**AI 应用场景**

- 处理大向量库（百万级 chunk）：不能一次性 load 进内存
- 流式处理数据集：读一行、处理一行、输出一行

**核心语法**

| 写法 | 名称 | 行为 |
|------|------|------|
| `[x for x in data]` | 列表推导式 | 立即全部计算，占内存 |
| `(x for x in data)` | 生成器表达式 | 惰性计算，用到才算 |

In [1]:
import sys

big_range = range(100_000_000)

list_demo = [x for x in range(1000)]   # 列表：立即算完，占内存
gen = (x * x for x in big_range)       # 生成器：不管数据多大，都只占一点点

print(f"列表推导式（1000 个元素）占用: {sys.getsizeof(list_demo)} 字节")
print(f"生成器表达式（1 亿个元素）占用: {sys.getsizeof(gen)} 字节")

列表推导式（1000 个元素）占用: 8856 字节
生成器表达式（1 亿个元素）占用: 200 字节


In [2]:
# 惰性求值：只取前 5 个，后面 1000 万个数根本没计算
gen2 = (x * x for x in range(10_000_000) if x % 2 == 0)

for i, val in enumerate(gen2):
    if i >= 5:
        print("  ...（后面的按需继续取，不占内存）")
        break
    print(f"第 {i + 1} 个偶数平方: {val}")

第 1 个偶数平方: 0
第 2 个偶数平方: 4
第 3 个偶数平方: 16
第 4 个偶数平方: 36
第 5 个偶数平方: 64
  ...（后面的按需继续取，不占内存）


## 处理大文件：yield 一次读一行

先定位知识库文件（从当前目录逐级向上找，无论 Notebook 从哪里启动都能找到）：

In [3]:
from pathlib import Path
from typing import Iterable

def find_kb() -> Path:
    # 从当前目录逐级向上找 04_rag/knowledge_base.txt
    for folder in [Path.cwd(), *Path.cwd().parents]:
        candidate = folder / "04_rag" / "knowledge_base.txt"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("找不到知识库文件")

def lazy_lines(file_path: Path) -> Iterable[str]:
    # 处理大文件的正确姿势：一次只读一行
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            yield line.strip()

kb = find_kb()
print(f"知识库位置: {kb}")
print()

for i, line in enumerate(lazy_lines(kb)):
    if i >= 3:
        print("...（后面的行按需继续读，不占内存）")
        break
    print(f"第 {i + 1} 行: {line[:40]}")

知识库位置: /Users/linmanyun/git-workspace/myFirstAgent/04_rag/knowledge_base.txt

第 1 行: # 幻境冒险游戏攻略手册（内部测试版 v0.9）
第 2 行: 
第 3 行: ## 第一章 新手入门
...（后面的行按需继续读，不占内存）


## 💡 小结

- 列表推导式：**先付钱后吃饭**（内存立即占用）
- 生成器表达式：**吃一口付一口**（惰性计算）
- Agent 项目处理海量 chunk / 长对话历史时，必须用生成器思维